# Test
Si addestri il regressore desiderato utilizzando gli iperparametri trovati nell'esercizio precedente. Il codice contenuto nella cella seguente userà tale regressore per predire la potenza prodotta nell'ora successiva a partire dai pattern del dataset di test. I valori predetti verranno memorizzati su un file di testo che dovrà essere caricato sul sito della competizione per misurarne l'RMSE.

In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from pre_processing import extract_datetime_feature, augment_celsius, pre_process,augment_cycle_variable,get_dataset_features
from matplotlib import pyplot as plt

In [ ]:
train_path = './DBs/SolarPark/train.txt'
test_path = './DBs/SolarPark/test.txt'
result_path = 'final.txt'

useless_column=["Date","year","day_part","Unnamed: 0","notes","note"]
category = ["day_of_week","day_of_year","day_part","is_year_start","is_month_start","is_quarter_start","is_month_end","is_weekend"]


In [ ]:
train_dataframe,train_pure_dataframe = pre_process(train_path,d=True,extension=True)
zeros = train_dataframe[(train_pure_dataframe["P (kW)"] == 0.0) & (train_dataframe["Ta (C)"] == 0.0) & (train_dataframe["Tm (C)"] == 0.0)]
train_dataframe.boxplot(column='P (kW)', by='Time Frame', figsize=(5, 5))

In [ ]:
for column in train_dataframe:
    if column in category:
        train_dataframe[column] =  train_dataframe[column].astype('category')
        train_pure_dataframe[column] =  train_pure_dataframe[column].astype('category')
    elif column == "Date":
        continue
    else:
        train_dataframe[column] =  train_dataframe[column].astype(np.float64)
#         train_pure_dataframe[column] =  train_pure_dataframe[column].astype(np.float64)
# train_dataframe["P (hp)"] = train_dataframe.apply(lambda x: x["P (kW)"] * 1.3151, axis=1)
train_dataframe["Ta (F)"] = train_dataframe.apply(lambda x: (9 / 5) * x["Ta (C)"] + 32, axis=1)
train_dataframe["Tm (F)"] = train_dataframe.apply(lambda x: (9 / 5) * x["Tm (C)"] + 32, axis=1)
# train_pure_dataframe["P (hp)"] = train_pure_dataframe.apply(lambda x: x["P (kW)"] * 1.3151, axis=1)
train_pure_dataframe["Ta (F)"] = train_pure_dataframe.apply(lambda x: (9 / 5) * x["Ta (C)"] + 32, axis=1)
train_pure_dataframe["Tm (F)"] = train_pure_dataframe.apply(lambda x: (9 / 5) * x["Tm (C)"] + 32, axis=1)
# train_dataframe["Tm (F)"] = train_dataframe.apply(lambda x: (9 / 5) * x["Tm (C)"] + 32, axis=1)
train_pure_dataframe = train_pure_dataframe.drop(columns="day_part")
# train_pure_dataframe = train_pure_dataframe.drop(columns="P (kW)")
feature_x, feature_y = get_dataset_features(train_dataframe, "P (kW)", useless_column)
feature_pure_x, feature_pure_y = get_dataset_features(train_pure_dataframe, "P (kW)", useless_column=useless_column)


test_dataframe = pd.read_csv(test_path)

test_dataframe = extract_datetime_feature(df=test_dataframe, date_column='Date', remove=False, is_timestamp=True)
test_dataframe = augment_celsius(df=test_dataframe, replace=False, columns=["Ta (C)", "Tm (C)"])
test_dataframe = augment_cycle_variable(df=test_dataframe, replace=False, columns=["Time Frame", "month", "day",
                                                                         "hour", "minute", "day_of_year",
                                                                         "day_of_week"])
test_dataframe = test_dataframe.drop(columns="day_part")
test_dataframe["Ta (F)"] = test_dataframe.apply(lambda x: (9 / 5) * x["Ta (C)"] + 32, axis=1)
test_dataframe["Tm (F)"] = test_dataframe.apply(lambda x: (9 / 5) * x["Tm (C)"] + 32, axis=1)
for column in test_dataframe:
    if column in category:
        test_dataframe[column] =  test_dataframe[column].astype('category')
    elif column == "Date":
        continue
    else:
        test_dataframe[column] =  test_dataframe[column].astype(np.float64)

test_x, _ = get_dataset_features(test_dataframe, "P (kW)", useless_column=useless_column)



In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

predict_ahead = True

if predict_ahead:
    # Sfasa i dati: le prediction sono quelle dell'ora successiva
    data_x = feature_x[0:-1,:]
    data_y = feature_y[1:]
    data_pure_x = feature_pure_x[0:-1,:]
    data_pure_y = feature_pure_y[1:]
else:
    data_x = feature_x
    data_y = feature_y
    data_pure_x = feature_pure_x
    data_pure_y = feature_pure_y


In [ ]:
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.linear_model import LinearRegression

model_params = {
    'RandomForestRegressor': {
        'model': RandomForestRegressor(),
        'params' : {
            'n_estimators' : [300,500,1000,1500,2000], # 300, 500, 700], #numero di alberi utilizzati (maggiore incrementa le performance e permette predizioni più stabili) #300,500,700,1000
            'max_depth' : [3,4,5,6,7,8,9,10,11,12,13,14,15], # The maximum depth of the tree. e.g. 3,5,7,10
            "max_features": [" sqrt", " log2", None], # e.g. [“sqrt”, “log2”, None]
            "min_samples_leaf": [1,2,3,4,5,6,7,], # istanze dei nodi foglia ---default 1
            "min_samples_split": [2,3,4,5,5,6,7], # istanze dei nodi interni ---default 2
            'criterion' : ['squared_error','absolute_error','poisson'], # ---default 'squared_error', then 'absolute_error', 'poisson'
            'n_jobs': [-1],
        }
    },
    'LinearRegressor': {
        'model': LinearRegression(),
        'params' : {
            'fit_intercept': [True,False], # if False, data is expected to be centered
            'copy_X': [True], # If True, X will be copied; else, it may be overwritten.
            'n_jobs': [-1],
            'positive': [False,True] # When set to True, forces the coefficients to be positive
        }
    }
}

votingRegressorModel = VotingRegressor([
    ('rf', RandomForestRegressor(**model_params['RandomForestRegressor']['params'])),
    ('lr', LinearRegression(**model_params['LinearRegressor']['params']))
])

In [ ]:
sc = RobustScaler()
data_pure_x = sc.fit_transform(data_pure_x)
test_x = sc.transform(test_x)

In [ ]:
# NOT USED!!!!!!!
# from sklearn.model_selection import GridSearchCV, ShuffleSplit
#
# test_size = [0.25]
# n_split = [9]
# scores = []
# for data_test_size in test_size: #0.2,0.25,0.3
#     for split in n_split: #, 2, 3, 4, 5, 6, 7, 8, 9, 10
#         cross_val = ShuffleSplit(n_splits=split, test_size=data_test_size, random_state=1234)
#
#         for model_name, mp in model_params.items():
#             grid = GridSearchCV(estimator=mp['model'],
#                                 param_grid=mp['params'],
#                                 cv=cross_val,
#                                 verbose=2,
#                                 return_train_score=False,
#                                 scoring='neg_mean_squared_error')
#             grid.fit(data_x,data_y)
#             scores.append({
#                 'data_test_size': data_test_size,
#                 'split': split,
#                 'model': model_name,
#                 'best_score': grid.best_score_,
#                 'mean_test_score': grid.cv_results_['mean_test_score'],
#                 'best_params': grid.best_params_,
#                 'grid': grid,
#             })
#
#         df = pd.DataFrame(scores, columns=['data_test_size', 'split', 'model', 'best_score', 'mean_test_score',
#                                            'best_params', 'grid'])

In [ ]:
from sklearn.model_selection import GridSearchCV, ShuffleSplit

test_size = [0.25]
n_split = [9]
scores = []
df_pure = pd.DataFrame()
for data_test_size in test_size: #0.2,0.25,0.3
    for split in n_split: #, 2, 3, 4, 5, 6, 7, 8, 9, 10
        cross_val = ShuffleSplit(n_splits=split, test_size=data_test_size, random_state=1234)

        for model_name, mp in model_params.items():
            grid = GridSearchCV(estimator=mp['model'],
                                param_grid=mp['params'],
                                cv=cross_val,
                                verbose=2,
                                return_train_score=False,
                                scoring='neg_mean_squared_error',)

            grid.fit(data_pure_x,data_pure_y)
            scores.append({
                'data_test_size': data_test_size,
                'split': split,
                'model': model_name,
                'best_score': grid.best_score_,
                'mean_test_score': grid.cv_results_['mean_test_score'],
                'best_params': grid.best_params_,
                'grid': grid,
            })

        df_pure = pd.DataFrame(scores, columns=['data_test_size', 'split', 'model', 'best_score', 'mean_test_score',
                                           'best_params', 'grid'])

In [ ]:
predictions_pure = pd.DataFrame()
for idx, row in df_pure.iterrows():
    forest = RandomForestRegressor(**row['best_params'])
    predictions_pure[f"grid_{idx}"] = forest.fit(data_pure_x,data_pure_y).predict(test_x)
    print(f"grid_{idx} completed!")

In [ ]:
predictions_pure['prediction_final'] = predictions_pure.mean(axis=1)

In [ ]:
train_dataframe.boxplot(column='P (kW)', by='Time Frame', figsize=(5, 5))
plt.show()
test_dataframe['prediction_final'] = predictions_pure['prediction_final']
test_dataframe.boxplot(column='prediction_final', by='Time Frame', figsize=(10, 10))
plt.show()

In [ ]:
np.savetxt(result_path, predictions_pure)
print('Ok')